# Классификация проектов с помощью линейной регрессии

In [1]:
# install libraries

%pip install numpy==1.23.5
%pip install typer==0.9.4
%pip install torch==2.0.1
%pip install transformers==4.34.0
%pip install sentence-transformers==3.0.0
%pip install spacy==3.5.4
%pip install tensorflow==2.12.0
%pip install torchtext==0.15.2
%pip install nltk==3.7
%pip install scipy==1.15.3
%pip install gensim==4.4.0
%pip install xgboost==1.7.6
%pip install catboost

%pip check

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 23.0.1 -> 26.1.1
[notice] To update, run: python3 -m pip install --upgrade pip
Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 23.0.1 -> 26.1.1
[notice] To update, run: python3 -m pip install --upgrade pip
Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 23.0.1 -> 26.1.1
[notice] To update, run: python3 -m pip install --upgrade pip
Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 23.0.1 -> 26.1.1
[notice] To update, run: python3 -m pip install --upgrade pip
Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 23.0.1 -> 26.1.1
[notice] To update, run: python3 -m pip install --up

In [3]:
# library deps
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import nltk
import pandas as pd

from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, train_test_split
from sklearn.preprocessing import LabelEncoder
from tqdm import tqdm
from gensim.models.word2vec import Word2Vec
from xgboost import XGBClassifier
from catboost import CatBoostClassifier, Pool

## Загрузка данных

In [ ]:
Labels = [
    "Автомобильные дороги",
    "Водоотведение",
    "Водопроводы",
    "Газоны дорожки",
    "Газопроводы",
    "Горные выработки",
    "Железнодорожные пути",
    "Заводы фабрики",
    "Здания",
    "Инженерное обеспечение",
    "Инфраструктура наземного электротранспорта",
    "Линии электропередачи",
    "Метрополитены",
    "Мосты и тоннели",
    "Наружное освещение",
    "Нефтепроводы",
    "Сооружения",
    "Теплопроводы",
    "Технологические установки",
]

ColumnNames = ["id", "project_name", "label"]


def load_labeled_data(path):
    labeled_dataframes = [
        pd.read_csv(f"{path}//{label}.csv", names=ColumnNames, header=0)
        for label in tqdm(Labels)
    ]
    result_df = pd.concat(labeled_dataframes)
    result_df["project_name"] = result_df["project_name"].str.strip('"')
    return result_df


def load_unlabeled_data(path):
    return pd.read_csv(path, sep=";", encoding="utf-8", nrows=200000, names=["id", "project_name"])


# raw_df = load_unlabeled_data(f'../Data/Реестр 2022-2024 clean.csv')
# raw_df

## Разделение данных на тестовую и обучающую выборки

In [ ]:
def data_train_test_split(data, labels):
    assert len(data) == len(
        labels
    ), "Размеры списков данных и результатов разметки не совпадают"
    le = LabelEncoder()
    le.fit(labels)
    y = le.transform(labels)
    return train_test_split(data, y, test_size=0.2, random_state=42)

## Способы векторизации

In [ ]:
def vectorize_words_with_word2vec(sentences, vector_size):
    nltk.download("punkt")
    tokenized_sentences = [
        nltk.tokenize.word_tokenize(text.lower(), language="russian")
        for text in tqdm(sentences)
    ]
    sentence_vectors = Word2Vec(
        tokenized_sentences,
        workers=8,
        vector_size=vector_size,
        min_count=3,
        window=5,
        epochs=15,
    )
    return sentence_vectors


def vectorize_with_word2vec(sentences, vector_size):
    result = []
    for word in word_tokenize(text.lower()):
        if word in model_tweets.wv:
            result.append(model_tweets.wv[word])

    if len(result):
        result = np.average(result, axis=0)
    else:
        result = np.zeros(300)
    return result

In [ ]:
# Вернет матрицу размера (len(sentences, 1024)
def vectorize_with_sentence_transformer(model_name, sentences):
    model = SentenceTransformer(model_name)
    return model.encode(sentences.to_numpy())

## Функции оптимизации с помощью Grid search и Random search

In [ ]:
def find_best_model_gs(X_train, y_train, estimator, param_grid):
    grid_search = GridSearchCV(
        estimator=estimator,
        param_grid=param_grid,
        scoring="accuracy",
        cv=3,
        n_jobs=-1
    )
    grid_search.fit(X_train, y_train)
    return grid_search.best_estimator_, grid_search.best_params_

def find_best_model_rs(X_train, y_train, estimator, param_dist, n_iter):
    random_search = RandomizedSearchCV(
        estimator=estimator,
        param_distributions=param_dist,
        n_iter=n_iter,
        scoring="accuracy",
        cv=3,
        n_jobs=-1
    )
    random_search.fit(X_train, y_train)
    return random_search.best_estimator_, random_search.best_params_

## Классификация проектов

### Загрузим данные

In [ ]:
df = load_labeled_data("../Data/Reestr/Размеченные")
df

### Векторизация

In [ ]:
# Векторизация WordToVek
# word2vec_vectors = vectorize_with_word2vec(df['project_name'], 300)

# word2vec_vectors.wv.most_similar('мост')

In [ ]:
# Векторизация с помощью SentenceTransformer с использованием модели 'sberbank-ai/sbert_large_nlu_ru'
sbert_vectors = vectorize_with_sentence_transformer(
    "sberbank-ai/sbert_large_nlu_ru", df["project_name"]
)

### Разделeние на тестовую и обучающую выборки

In [ ]:
X_train, X_test, y_train, y_test = data_train_test_split(sbert_vectors, df["label"])

### Логистическая регрессия

#### Параметры модели

In [ ]:
log_reg_param_grid = {
        "C": [0.05, 0.1, 1, 10, 50],
        "penalty": ["l1", "l2", 'elasticnet'],
        "solver": ["liblinear", "saga"],
        "max_iter": [500, 1000]}

log_reg_test_param_grid = {
        "C": [1],
        "penalty": ["l1"],
        "solver": ["liblinear"],
        "max_iter": [100]}

 #### Вариант с векторизацией sber sentence transformer

In [ ]:
# Выбор лучшей модели
lr_initial_model = LogisticRegression(random_state=42)
#lr_best_model, lr_best_params = find_best_model_gs(X_train, y_train, lr_initial_model, log_reg_test_param_grid)
lr_best_model, lr_best_params = find_best_model_rs(X_train, y_train, lr_initial_model, log_reg_param_grid, 20)
print("Лучшие параметры: ", lr_best_params)

lr_best_model.fit(X_train, y_train)
y_pred = lr_best_model.predict(X_test)

print("Точность:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

### XGBoost

#### Параметры модели

In [ ]:
xgb_test_param_grid = {"n_estimators": [50], "max_depth": [6], "learning_rate": [1]}

xgb_param_grid = {
    "n_estimators": [50, 80, 100, 150, 200],
    "max_depth": [4, 6, 8],
    "learning_rate": [0.05, 0.1, 0.3, 0.5, 1],
}

#### Вариант с векторизацией sber sentence transformer

In [ ]:
xgb_initial_model = XGBClassifier(use_label_encoder=False, n_jobs=-1, eval_metric="logloss", tree_method="gpu_hist", random_state=42)
#xgb_best_model, xgb_best_params = find_best_model_gs(X_train, y_train, xgb_initial_model, xgb_test_param_grid)
xgb_best_model, xgb_best_params = find_best_model_rs(X_train, y_train, xgb_initial_model, xgb_test_param_grid, 1)
xgb_best_model.fit(X_train, y_train)
y_pred = xgb_best_model.predict(X_test)

print("Лучшие параметры: ", xgb_best_params)
print("Точность:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

### CatBoost

#### Параметры модели

In [ ]:
cb_param_grid = {
        "learning_rate": [0.01, 0.05, 0.1, 0.5],
        "depth": [3, 4, 6, 8],
        "l2_leaf_reg": [1, 2, 4, 8, 10],
        "iterations": [50, 100, 300, 500],
        "random_strength": [0.5, 1, 2.0],
        "bagging_temperature": [0.8, 1.0, 1.2]}

cb_test_param_grid = {
        "learning_rate": [0.1],
        "depth": [6],
        "l2_leaf_reg": [3],
        "iterations": [50],
        "random_strength": [1],
        "bagging_temperature": [1.0]}

#### Вариант с векторизацией sber sentence transformer

In [ ]:
cb_initial_model = CatBoostClassifier(iterations=100, loss_function='MultiClass', random_seed=42)
#cb_best_model, cb_best_params = find_best_model_rs(X_train, y_train, cb_initial_model, cb_test_param_grid)
cb_best_model, cb_best_params = find_best_model_rs(X_train, y_train, cb_initial_model, cb_test_param_grid, 1)

#cb_best_model.fit(X_train, y_train)
y_pred = cb_best_model.predict(X_test)

print("Лучшие параметры: ", cb_best_params)
print("Точность:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))